<a href="https://colab.research.google.com/github/hpatel1933/AAI2025/blob/main/Exercise_2_ReACT_Code_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
def grade_student(scores):
    # Validate the input before calculating the average.
    if not isinstance(scores, list):
        return None, "Input must be a list."
    if not scores:
        return None, "Scores list cannot be empty."
    for score in scores:
        if not isinstance(score, (int, float)):
            return None, "All scores must be numeric."
        if not 0 <= score <= 100:
            return None, "Scores must be between 0 and 100."
    average = round(sum(scores) / len(scores), 2)
    if average >= 90:
        letter = "A"
    elif average >= 80:
        letter = "B"
    elif average >= 70:
        letter = "C"
    elif average >= 60:
        letter = "D"
    else:
        letter = "F"
    return average, letter

print("NORMAL:", grade_student([90, 85, 95]))
print("EMPTY LIST:", grade_student([]))
print("INVALID SCORE:", grade_student([90, 105, 80]))
assert grade_student([90, 85, 95]) == (90.0, "A")
assert grade_student([])[1] == "Scores list cannot be empty."
assert grade_student([90, 105, 80])[1] == "Scores must be between 0 and 100."
print("SEPARATE CODE CELL VALIDATION: PASS")

NORMAL: (90.0, 'A')
EMPTY LIST: (None, 'Scores list cannot be empty.')
INVALID SCORE: (None, 'Scores must be between 0 and 100.')
SEPARATE CODE CELL VALIDATION: PASS


# Exercise 2: ReACT-Style Python Code Generation

Goal: generate a student average and letter-grade program using Plan, Generate, Run, Observe, Fix, and Final stages. Tools: Google Colab and Python. The code below shows the prompt, working implementation, normal tests, and edge-case tests.

In [ ]:
!pip -q install -U openai
from openai import OpenAI
from google.colab import userdata
api_key = userdata.get("OPENROUTER_API_KEY")
assert api_key, "Add a Colab Secret named OPENROUTER_API_KEY and enable notebook access."
client = OpenAI(api_key=api_key, base_url="https://openrouter.ai/api/v1")
MODEL = "openai/gpt-4o-mini"
definition = "Create grade_student(scores) returning (average, letter), rejecting empty lists and scores outside 0-100."
plan_prompt = "Plan validation, average calculation, grade mapping, and edge-case tests for this task: " + definition
plan = client.chat.completions.create(model=MODEL, messages=[{"role":"user","content":plan_prompt}], temperature=0).choices[0].message.content.strip()
generate_prompt = "Return only Python code between CODE_START and CODE_END. Define grade_student(scores). Return (average, letter) for valid numeric scores 0-100; return (None, message) for empty or invalid input. No imports or explanation. Task: " + definition + " Plan: " + plan
draft_response = client.chat.completions.create(model=MODEL, messages=[{"role":"user","content":generate_prompt}], temperature=0).choices[0].message.content.strip()
draft_code = draft_response.replace(chr(96), "").replace("CODE_START", "").replace("CODE_END", "").strip()
draft_code = draft_code[6:].lstrip() if draft_code.startswith("python") else draft_code
namespace = {}
observation = ""
try:
    exec(draft_code, namespace)
    observation = "Draft normal=" + str(namespace["grade_student"]([90, 85, 95])) + "; empty=" + str(namespace["grade_student"]([]))
except Exception as error:
    observation = "Draft failed: " + type(error).__name__ + ": " + str(error)
fix_prompt = "Return only corrected Python code between CODE_START and CODE_END. Define grade_student(scores). Return (average, letter) for valid numeric scores 0-100; return (None, message) for empty or invalid input. No imports or explanation. Task: " + definition + " Draft: " + draft_code + " Observation: " + observation
fix_response = client.chat.completions.create(model=MODEL, messages=[{"role":"user","content":fix_prompt}], temperature=0).choices[0].message.content.strip()
final_code = fix_response.replace(chr(96), "").replace("CODE_START", "").replace("CODE_END", "").strip()
final_code = final_code[6:].lstrip() if final_code.startswith("python") else final_code
final_namespace = {}
exec(final_code, final_namespace)
normal_result = final_namespace["grade_student"]([90, 85, 95])
empty_result = final_namespace["grade_student"]([])
invalid_result = final_namespace["grade_student"]([90, 105, 80])
print("OPENAI-COMPATIBLE PROVIDER: OpenRouter")
print("MODEL:", MODEL)
print("\nPLAN:\n", plan)
print("\nGENERATE RESPONSE:\n", draft_response)
print("\nRUN/OBSERVE:\n", observation)
print("\nFIX RESPONSE:\n", fix_response)
print("\nFINAL CODE:\n", final_code)
print("\nFINAL TESTS:", normal_result, empty_result, invalid_result)
assert normal_result[1] == "A" and empty_result[0] is None and invalid_result[0] is None
print("\nREACT VALIDATION: PASS - live OpenRouter plan, generation, observation, fix, and tests completed.")

OPENAI-COMPATIBLE PROVIDER: OpenRouter
MODEL: openai/gpt-4o-mini

PLAN:
 To create a function `grade_student(scores)` that calculates the average score and maps it to a letter grade, we need to consider several aspects, including input validation, average calculation, grade mapping, and edge-case tests. Below is a structured plan for implementing this function.

### Function Implementation Plan

1. **Input Validation**:
   - Check if the input `scores` is a list.
   - Reject empty lists and raise an appropriate error.
   - Ensure all scores are numeric (integers or floats).
   - Check that all scores are within the range of 0 to 100. If any score is outside this range, raise an appropriate error.

2. **Average Calculation**:
   - Calculate the average of the valid scores.

3. **Grade Mapping**:
   - Map the average score to a letter grade based on the following scale:
     - A: 90 - 100
     - B: 80 - 89
     - C: 70 - 79
     - D: 60 - 69
     - F: 0 - 59

4. **Return Values**:
   - R